# 10C · Value at Risk & the Fat-Tail Payoff
### Financial Analytics — Module 10 · Lab 2

Module 3 showed you the most important picture in quantitative finance — real returns have **fat tails** the bell curve denies. This notebook is where that picture finally costs (or saves) money:

1. **VaR by simulation** — "how much could we lose on a bad day?"
2. **The normal-world lie** — simulate with the bell curve, understate the danger
3. **The bootstrap fix** — resample *actual history*, keep the fat tails
4. Scaling to 10 days, and what VaR honestly is (and isn't)

> 🛡️ **Bias check:** VaR from history assumes the sampled past spans the relevant futures — a REGIME bet, stated. Our window includes both calm and choppy regimes (good); it includes no 2008-scale event (limitation, stated). Survivorship: index data, none. Look-ahead: none.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

BASE = "data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
rets = px["close"].pct_change().dropna().values
PORTFOLIO = 10_00_00_000    # Rs 10 crore book
print(f"{len(rets)} daily returns | book size Rs {PORTFOLIO/1e7:.0f} cr")

---
## 1. What VaR is, in one honest sentence

> **95% 1-day VaR = the loss so bad that only 5% of days are worse.**

Not the worst case. Not a guarantee. A *percentile of the loss distribution* — which is why Monte Carlo (and its cousin, historical resampling) is the natural way to compute it: build the distribution, read the percentile.

## 2. Two worlds, one book

**World A — the bell-curve world:** simulate tomorrow 100,000 times drawing from a normal with history's mean and std.
**World B — the bootstrap world:** simulate tomorrow by drawing *actual past days* at random, with replacement. No distribution assumed; the fat tails come along free.

In [ ]:
N = 100_000
mu, sigma = rets.mean(), rets.std()

sim_normal = rng.normal(mu, sigma, N)                    # World A
sim_boot   = rng.choice(rets, N, replace=True)           # World B: history, reshuffled

def var_es(sims, level=0.95):
    losses = -sims * PORTFOLIO
    var = np.quantile(losses, level)
    es  = losses[losses >= var].mean()                   # Expected Shortfall: the average of the bad tail
    return var, es

for name, sims in [("Normal (bell) world", sim_normal), ("Bootstrap (real) world", sim_boot)]:
    v95, e95 = var_es(sims, 0.95)
    v99, e99 = var_es(sims, 0.99)
    print(f"{name:<24} 95% VaR Rs {v95/1e5:6.1f} L | 99% VaR Rs {v99/1e5:6.1f} L | 99% ES Rs {e99/1e5:6.1f} L")

In [ ]:
# WHERE the two worlds disagree: deep in the tail
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(-0.06, 0.06, 120)
ax.hist(sim_boot, bins=bins, density=True, alpha=0.55, color="#DC2626", label="bootstrap (real tails)")
ax.hist(sim_normal, bins=bins, density=True, alpha=0.55, color="#2563EB", label="normal (bell)")
ax.set_yscale("log")                                     # log scale: the tails become visible
ax.set_title("The two worlds on a LOG scale - identical middles, divergent tails", loc="left", fontweight="bold")
ax.set_xlabel("daily return"); ax.legend(); plt.tight_layout(); plt.show()

for thr in [0.02, 0.03, 0.04]:
    pn = (sim_normal < -thr).mean(); pb = (sim_boot < -thr).mean()
    ratio = pb/pn if pn > 0 else float('inf')
    print(f"P(loss worse than {thr:.0%}): normal {pn:.4%} vs real {pb:.4%}   -> reality is {ratio:,.1f}x more dangerous")

**There's Module 3's chart, now with a price tag.** At the 95% level the two worlds roughly agree — the bell curve is fine in the middle. Go deeper — 3%, 4% loss days — and reality is *multiples* more likely than the normal world admits. Every risk system that assumed normality (and before 2008, most did) was **most wrong exactly where being wrong costs most**. The bootstrap costs one line of numpy and keeps the truth.

*(Also note **Expected Shortfall** in the table: "IF we breach VaR, how bad is the average breach?" It answers the question VaR dodges, and it's what modern regulation increasingly asks for — Basel's market-risk rules moved from VaR to ES. Two lines of code apart; a philosophy apart.)*

---
## 3. Ten days out: simulate, don't shortcut

In [ ]:
# The 10-day VaR: the textbook shortcut multiplies 1-day VaR by sqrt(10). Monte Carlo doesn't need the shortcut -
# simulate 10-day paths by drawing 10 real days and compounding:
paths10 = np.take(rets, rng.integers(0, len(rets), (N, 10)))
ret10 = np.prod(1 + paths10, axis=1) - 1
losses10 = -ret10 * PORTFOLIO

v1 = np.quantile(-sim_boot*PORTFOLIO, .99)
v10_sim = np.quantile(losses10, .99)
print(f"99% VaR, 1-day  : Rs {v1/1e5:.1f} L")
print(f"99% VaR, 10-day : Rs {v10_sim/1e5:.1f} L (simulated)")
print(f"sqrt(10) rule   : Rs {v1*np.sqrt(10)/1e5:.1f} L")
print(f"\nClose but not identical - the rule assumes i.i.d. normal days; simulation just counts.")
print("When assumptions and counting disagree, counting wins. That IS the Monte Carlo creed.")

## 4. What VaR honestly is — the three sentences

Present VaR with these attached, always:

1. **It's a percentile, not a ceiling** — 99% VaR is *exceeded* about 2–3 times a year by design; a breach is not a model failure.
2. **It's history-shaped** — a VaR from a calm window is a calm-window VaR (the regime bet from the bias check, restated).
3. **It says nothing about how bad breaches get** — that's Expected Shortfall's job. Quote them together.

### ✏️ Exercises
1. **Regime-split VaR:** compute bootstrap 99% VaR separately from the calm (2021–22) and choppy (2023+) windows. How different? Which would you report, and with what sentence?
2. **The clustering wrinkle:** our 10-day bootstrap drew days *independently*, but 9B proved volatility clusters. Redo the 10-day paths by drawing 10 *consecutive* real days (a random start, then a block). Does block-sampling widen the 99% tail? (This is the "block bootstrap" — one line different, honesty preserved.)
3. **Backtest the VaR:** walk through history — each day, compute 95% VaR from the previous 250 days, then check whether the NEXT day breached it. Count breaches. A well-calibrated 95% VaR breaches ~5% of days. Does ours? In which periods do breaches cluster — and what does that clustering tell you?

---
## Lab 2 complete

Random walks from coin flips · √t derived by counting · the cone built from scratch · a retirement plan honestly stress-tested · sequence risk demonstrated · NPV as a distribution with P(loss) · VaR and ES with real tails · and the creed: **when assumptions and counting disagree, counting wins.** **Badge: Futurist 🎲**

*AI disclosure: ______*

In [ ]:
# workspace
